In [1]:
%load_ext autoreload
%autoreload 2

In [30]:
import tensorflow as tf
from util import yolo_filter_boxes, yolo_non_max_suppression, yolo_eval, read_classes, read_anchors
from model_torch import YOLOv2, train_model, Darknet19, wrap_yolo_loss, YoloV2GridTransform
import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from dataset import VOCDataset, VOC_CLASSES
import matplotlib.pyplot as plt


In [ ]:
tf.random.set_seed(10)
box_confidence = tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([19, 19, 5, 4], mean=1, stddev=4, seed = 1)
box_class_probs = tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = 0.5)
print("\n")
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.shape))
print("boxes.shape = " + str(boxes.shape))
print("classes.shape = " + str(classes.shape))

In [ ]:
tf.random.set_seed(10)
scores = tf.random.normal([54,], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([54, 4], mean=1, stddev=4, seed = 1)
classes = tf.random.normal([54,], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_non_max_suppression(scores, boxes, classes)

print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

In [ ]:
tf.random.set_seed(10)
yolo_outputs = (tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1))
scores, boxes, classes = yolo_eval(yolo_outputs)
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

In [5]:
class_names = read_classes("model_data/coco_classes.txt")
print(class_names)
anchors = read_anchors("model_data/yolo_anchors.txt")
model_image_size = (608, 608) # Same as yolo_model input layer size
print(anchors)
num_classes = len(VOC_CLASSES)
num_anchors = len(anchors)

['car']
anchors ['0.57273', ' 0.677385', ' 1.87446', ' 2.06253', ' 3.33843', ' 5.47434', ' 7.88282', ' 3.52778', ' 9.77052', ' 9.16828']
[[0.57273  0.677385]
 [1.87446  2.06253 ]
 [3.33843  5.47434 ]
 [7.88282  3.52778 ]
 [9.77052  9.16828 ]]


In [6]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [7]:
data_transforms = transforms.Compose([
    transforms.Resize((416,416)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAutocontrast(0.1),
    v2.ToImage(),                          # 1. Converts PIL Image to a Tensor image
    v2.ToDtype(torch.float32, scale=True), # 2. Converts to Float32 AND sca# les values to [0.0, 1.0]
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Always last
])
generators = torch.Generator().manual_seed(42)
batch_size = 64

dataset = VOCDataset(
    root="model_data/VOC2007",
    split_file=(
        "model_data/VOC2007/"
        "ImageSets/Main/trainval.txt"
    ),
    batch_size=batch_size,
    grid_shape=(13,13),
    anchors=anchors,
    transform=data_transforms,
)

print("dataset", dataset)
image, target = dataset[0]
print(image.size)
print(target)

# train_dataset = datasets.ImageFolder("model_data/VOC2007", transform=data_transforms)
train_dataset, val_dataset = random_split(dataset, [.8, .2], generator=generators)
val_dataset, test_dataset = random_split(val_dataset, [.5, .5], generator=generators)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(len(train_dataset), len(val_dataset), len(test_dataset))

# label_data [ 0.631       0.53333333 -0.16710549  0.76612021  1.          0.
#             label_data [ 0.434       0.024      -0.37782256 -0.37991716  1.          0.

dataset <dataset.VOCDataset object at 0x17fc76e40>
<built-in method size of Image object at 0x17fdcd5e0>
tensor([[[[ 0.4550,  0.5893,  0.4630,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.7020,  0.8200, -0.2890,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.8060,  0.2533, -0.1508,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000

In [ ]:
image, target = train_dataset[0]
print(type(image), type(target))

In [ ]:
# Convert the loader into an iterator
data_iter = iter(train_dataloader)

# Fetch the next single batch
images, labels = next(data_iter)

print("Single Batch Stats:")
print(f"Data Batch Shape: {images.shape}")
print(f"Label Batch Shape: {labels}")

In [8]:
learning_rate = 1e-3
weight_decay = 1e-4
epochs = 3 # use small epochs for transfer learning
# num_classes = len(class_names)

# model = CNNModel(num_classes).to(device)
model = YOLOv2(num_anchors, num_classes).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs, # number of epochs
    eta_min=1e-6
)

In [ ]:
# model = Darknet19()

# x = torch.randn(1, 3, 416, 416)
# with torch.no_grad():
#     y = model(x)

# print("output:", y.shape)

In [ ]:
# from torchinfo import summary

# summary(
#     model,
#     input_size=(1, 3, 416, 416)
# )
x = torch.tensor([[0, 0.97, 0],
                  [0.98,  0, 0.2]]).to(device)


# print(x[0, 2, 415, 415])
# y = model(x)
# print(y.shape)
# train_correct = (prediction == y).sum().item()
# print(train_correct)
# y = encode_archor(y, num_anchors, num_classes)
# print(y.shape)

# yolo_loss()



tensor(1, device='mps:0')
tensor(0, device='mps:0')


IndexError: index 2 is out of bounds for dimension 0 with size 2

In [ ]:
epochs = 3
loss_fn = wrap_yolo_loss(loss_weight=[1,1,1,1])
history = train_model(epochs,model, train_dataloader, val_dataloader, loss_fn, optimizer,scheduler, batch_size, device, num_classes)

train_loop-0 <class 'torch.Tensor'> <class 'torch.Tensor'>
stage1: torch.Size([64, 32, 208, 208])
stage2: torch.Size([64, 64, 104, 104])
stage3: torch.Size([64, 128, 52, 52])
stage4: torch.Size([64, 256, 26, 26])
stage5: torch.Size([64, 512, 26, 26])
pool5: torch.Size([64, 512, 13, 13])
stage6: torch.Size([64, 1024, 13, 13])
before passthrough torch.Size([64, 512, 26, 26])
after passthrough torch.Size([64, 256, 13, 13])

x after concatenate		: torch.Size([64, 1280, 13, 13])
x after stage7 #0	: torch.Size([64, 1024, 13, 13])
x after stage7 #1	: torch.Size([64, 125, 13, 13])
torch.Size([64, 13, 13, 5]) torch.Size([64, 13, 13, 5, 25])
loss:     nan  [   64/ 4009]
train_loop-1 <class 'torch.Tensor'> <class 'torch.Tensor'>
stage1: torch.Size([64, 32, 208, 208])
stage2: torch.Size([64, 64, 104, 104])
stage3: torch.Size([64, 128, 52, 52])
stage4: torch.Size([64, 256, 26, 26])
stage5: torch.Size([64, 512, 26, 26])
pool5: torch.Size([64, 512, 13, 13])
stage6: torch.Size([64, 1024, 13, 13])
befo

RuntimeError: The size of tensor a (64) must match the size of tensor b (13) at non-singleton dimension 1

In [26]:
weights_path = "save/yolov2.pth"
torch.save(model.state_dict(), weights_path)
print(f"Weights successfully saved to {weights_path}")

Weights successfully saved to save/yolov2.pth


In [29]:
state_dict = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)
print("Weights successfully loaded into model.")

Weights successfully loaded into model.


In [ ]:
print("Precision:", history["val_precision"])
print("Recall:", history["val_recall"])
print("F1:", history["val_f1"])

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15,5))
plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

NameError: name 'history' is not defined

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()